# SPARC Example 21: Querying the Corpus by Property

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

The flat CSV enables fast filtering by any galaxy property.
This example demonstrates practical query patterns researchers use:
- Find the largest galaxies
- Find galaxies with strong gas content
- Find galaxies with many data points (best-sampled RCs)
- Find galaxies by Hubble type

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import csv
import numpy as np

rows = []
with open('rotation_curve_corpus_v7_flat.csv') as f:
    for r in csv.DictReader(f):
        rows.append(r)

# Query 1: Largest galaxies by R_max
sparc = [r for r in rows if r['survey']=='SPARC' and r['r_max_kpc']]
largest = sorted(sparc, key=lambda x: float(x['r_max_kpc']), reverse=True)[:5]
print("Top 5 largest SPARC galaxies by R_max:")
for r in largest:
    print(f"  {r['galaxy']:<15} R_max={float(r['r_max_kpc']):.1f} kpc  "
          f"Vmax={r['vrot_max_kms']} km/s")

# Query 2: Most data points
best = sorted([r for r in sparc if r['n_points']],
              key=lambda x: int(float(x['n_points'])), reverse=True)[:5]
print("\nTop 5 best-sampled SPARC rotation curves:")
for r in best:
    print(f"  {r['galaxy']:<15} N={int(float(r['n_points']))} points")

# Query 3: Galaxies with negative Vgas rows (strong gas term)
gas_rich = [r for r in sparc if r.get('vgas_negative_rows')
            and float(r['vgas_negative_rows']) > 3]
print(f"\nSPARC galaxies with >3 negative Vgas rows (strong gas term): {len(gas_rich)}")
for r in gas_rich[:5]:
    print(f"  {r['galaxy']:<15} neg_Vgas_rows={r['vgas_negative_rows']}")